In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

#load in the cleaned test and training datasets from week 3
test_df = pd.read_csv("cleaned_test.csv")
train_df = pd.read_csv("cleaned_training.csv")

print(f"Test DataFrame shape: {test_df.shape}")
print(f"Training DataFrame shape: {train_df.shape}")

#feature_cols = ['LivingArea','BedroomsTotal','BathroomsTotalInteger',
#                      'LotSizeSquareFeet', 'zip_median_price', 'city_median_price' ]

feature_cols = [
    'LivingArea', 
    'BedroomsTotal', 
    'BathroomsTotalInteger',
    'LotSizeSquareFeet', 
    'zip_median_price', 
    'city_median_price',
    'bed_bath_ratio', 
    'property_age', 
    'district_median_price'
]

X_train = train_df[feature_cols].copy()
y_train = train_df['ClosePrice'].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df['ClosePrice'].copy()

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}  | y_test shape:  {y_test.shape}")

Test DataFrame shape: (12784, 10)
Training DataFrame shape: (71099, 10)
X_train shape: (71099, 9) | y_train shape: (71099,)
X_test shape:  (12784, 9)  | y_test shape:  (12784,)


Baseline Linear Regressor R² Scores (from week 4)

In [2]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

y_train_pred = linear_model.predict(X_train)
y_test_pred = linear_model.predict(X_test)

lr_train_results = r2_score(y_train, y_train_pred)
lr_test_results = r2_score(y_test, y_test_pred)
print('Linear Regression Results: \n')
print(f'Training R²: {lr_train_results:.4f}')
print(f'Test R²: {lr_test_results:.4f}')

Linear Regression Results: 

Training R²: 0.7923
Test R²: 0.7992


Decision Tree Regressor R² Scores

In [3]:
decision_tree = DecisionTreeRegressor()
decision_tree.fit(X_train, y_train)

y_train_pred = decision_tree.predict(X_train)
y_test_pred = decision_tree.predict(X_test)

dt_train_results = r2_score(y_train, y_train_pred)
dt_test_results = r2_score(y_test, y_test_pred)
print('Decision Tree Results: \n')
print(f'Training R²: {dt_train_results:.4f}')
print(f'Test R²: {dt_test_results:.4f}')

Decision Tree Results: 

Training R²: 0.9998
Test R²: 0.7365


Random Forest Regressor R² Results

In [4]:
random_forest = RandomForestRegressor()
random_forest.fit(X_train, y_train)

y_train_pred = random_forest.predict(X_train)
y_test_pred = random_forest.predict(X_test)

rf_train_results = r2_score(y_train, y_train_pred)
rf_test_results = r2_score(y_test, y_test_pred)
print('Random Forest Results: \n')
print(f'Training R²: {rf_train_results:.4f}')
print(f'Test R²: {rf_test_results:.4f}')

Random Forest Results: 

Training R²: 0.9809
Test R²: 0.8667


In [5]:
results = []
results.append({'Model': 'Linear Regression', 'Training R²': lr_train_results, 'Test R²': lr_test_results})
results.append({'Model': 'Decision Tree', 'Training R²': dt_train_results, 'Test R²': dt_test_results})
results.append({'Model': 'Random Forest', 'Training R²': rf_train_results, 'Test R²': rf_test_results})

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='Test R²', ascending=False).reset_index(drop=True)
print("\nModel Comparison Results:")
print(results_df)


Model Comparison Results:
               Model  Training R²   Test R²
0      Random Forest     0.980887  0.866671
1  Linear Regression     0.792313  0.799203
2      Decision Tree     0.999843  0.736548


Direct comparison to baseline model

In [6]:
baseline_r2 = 0.761132

rf_improvement = 0.836335 - baseline_r2
dt_improvement = 0.710998 - baseline_r2

print(f"Random Forest improvement: {rf_improvement:.4f}")
print(f"Decision Tree improvement: {dt_improvement:.4f}")

Random Forest improvement: 0.0752
Decision Tree improvement: -0.0501


#### Results:

- *Best Performing Model* - Random Forest Model
    - Builds multiple trees on various data subsets and features rather than memorizing patterns

- *Most Stable Model* - Linear Regression Model

- *Weaker Generalization Model* - Decision Tree Model:
    - Memorizes patterns on training data which doesn't generalize well to the test data

| Model | Train R² | Test R² | R² Difference | Strengths | Weaknesses |
| -------- | -------- | -------- | -------- | -------- | -------- |
| Random Forest Regressor | 0.978122  | 0.836335  | 0.141787  | Highest accuracy, captures complex relationships  | Slower training, harder to interpret  |
| Linear Regression  | 0.777826  | 0.761132  | 0.016694  | Stable, simple and fast  | Struggles to capture nonlinear patterns  |
| Decision Tree Regressor  | 0.998748  | 0.710998  | 0.28775  | Captures nonlinear patterns  | Overfits easily, unstable  |

#### Week 6 Results

| Model | Old Test R² | New Test R² | Improvement |
| -------- | -------- | -------- | -------- |
| Random Forest Regressor | 0.836335  | 0.865858  | 0.029523  |
| Linear Regression  | 0.761132  | 0.799203  | 0.038071  |
| Decision Tree Regressor  | 0.710998  | 0.744029  | 0.033031  |

##### Old Features
- `LivingArea`, `BedroomsTotal`, `BathroomsTotalInteger`, `LotSizeSquareFeet`, `zip_median_price`, `city_median_price`

##### New Features
- `property_age`: years since property was built (2026 - YearBuilt)
- `bed_bath_ratio`: bedrooms divided by bathrooms
- `district_median_price`: median ClosePrice per Unified School District

##### Conclusion

- All the models had improvement with the new feature set
    - Most Improvement: Linear Regression had the largest improvement of 0.038
    - Top Performing Model: Random Forest had the best Test R² score of 0.865858
- Creating the school district spatial layer (`district_median_price`) provided a tighter price baseline than ZIP codes and city medians alone, improving test accuracy across the models.
- Adding features like `bed_bath_ratio` and `property_age` helped models account for property condition and layout efficiency increasing model accuracy.